# **Linux Mint Mods**

**________________________________________________________________________________________________________________________________________________________________________________________________**

## Nemo Scripting
### Set Screenshot Dir
Set any folder as a default screenshot saving path:

**Step 1:** Create the file:

*`nano ~/.local/share/nemo/scripts/Set\ Screenshot\ Dir`*

**Step 2:** Paste the following script
```
#!/bin/bash


read -r TARGET_DIR <<< "$NEMO_SCRIPT_SELECTED_FILE_PATHS"

COUNT=$(printf '%s' "$NEMO_SCRIPT_SELECTED_FILE_PATHS" | grep -c .)

if [ "$COUNT" -eq 0 ]; then
    notify-send "Screenshot Dir" "⚠️ Please select a folder first."
    exit 1
fi
if [ "$COUNT" -gt 1 ]; then
    notify-send "Screenshot Dir" "⚠️ Please select only ONE folder."
    exit 1
fi

FILE_URI=$(python3 -c "from gi.repository import Gio; import sys; print(Gio.File.new_for_path(sys.argv[1]).get_uri())" "$TARGET_DIR")

gsettings set org.gnome.gnome-screenshot auto-save-directory "$FILE_URI"
gsettings set org.gnome.gnome-screenshot last-save-directory "$FILE_URI"

notify-send "Screenshot Dir ✅" "Now saving to:
$TARGET_DIR"
```

**Step 3:** Then make it executable 

*`chmod +x ~/.local/share/nemo/scripts/Set\ Screenshot\ Dir`*


**________________________________________________________________________________________________________________________________________________________________________________________________**

### Copy Files/Folders paths

**1. Install the clipboard tool:**

Open your terminal and install xclip (the standard clipboard tool for X11):

*`sudo apt update`*

*`sudo apt install xclip`*

**2. Create the action file:**

Create the directory for custom actions (if it doesn't exist) and open a new file:

*`mkdir -p ~/.local/share/nemo/actions`*

*`nano ~/.local/share/nemo/actions/copy-path.desktop`*

**3. Add the configuration:**

Paste the following code into the file:

```
[Nemo Action]
Name=Copy Path
Comment=Copy the full path to the clipboard
Exec=sh -c 'echo "%F" | xargs -n1 | xclip -selection clipboard'
Icon=edit-copy
Selection=any
Extensions=any;
```

**3. Restart Nemo**

Save the file (Ctrl+O, Enter, Ctrl+X) and restart Nemo:

`killall nemo`

`nemo &`

**________________________________________________________________________________________________________________________________________________________________________________________________**

## **Optimized Linux Mint settings / Removing slow defaults**

**1. Run these diagnostic commands**

Copy and paste these into your terminal:

Check Swappiness: default 60

*`cat /proc/sys/vm/swappiness`*

Check Power Managers:

```
systemctl status power-profiles-daemon --no-pager
systemctl status tlp --no-pager
```

Check Disk Caching: default 10, 20

```
cat /proc/sys/vm/dirty_background_ratio
cat /proc/sys/vm/dirty_ratio
```

**Fix Memory and Disk Caching**

Adjusts swappiness and dirty ratios

```
echo -e "\n# Custom Performance Tweaks" | sudo tee -a /etc/sysctl.conf
echo "vm.swappiness=10" | sudo tee -a /etc/sysctl.conf
echo "vm.dirty_background_ratio=5" | sudo tee -a /etc/sysctl.conf
echo "vm.dirty_ratio=10" | sudo tee -a /etc/sysctl.conf
sudo sysctl -p
```

**Install auto-cpufreq**

```
sudo apt install git
git clone https://github.com/AdnanHodzic/auto-cpufreq.git
cd auto-cpufreq
sudo ./auto-cpufreq-installer
```
Once the installer script finishes, activate the daemon to run automatically in the background:

*`sudo auto-cpufreq --install`*

To verify that the new power manager is actively doing its job, run:

*`sudo auto-cpufreq --stats`*

### Turn ON Turbo Boost

*`sudo nano /etc/auto-cpufreq.conf`*

Paste:

```
[charger]
governor = performance
turbo = auto

[battery]
governor = powersave
turbo = never
```

Restart: 

*`sudo systemctl restart auto-cpufreq`*

### Enabling Periodic TRIM for your NVMe SSD

```
sudo systemctl enable fstrim.timer
sudo systemctl start fstrim.timer
```

**________________________________________________________________________________________________________________________________________________________________________________________________**

### **Search Text from pdf and other files**

**Step 1: Install the tool**

*`sudo apt install pdfgrep`*

**Step 2: Make functions:**

1. *`nano ~/.bashrc`*
2. Paste at the bottom of the bashrc
```
find_pdf () {
  pdfgrep -r -n -i "$1" .
}

find_text () {
  grep -r -n -i "$1" .
}

find_code () {
  grep -r -n -i "$1" --include="*.py" --include="*.rpy" --include="*.js" --include="*.c" --include="*.cpp" --include="*.ipynb"  .
}
```
3. *`source ~/.bashrc`*
   
**Examples**
```
find_pdf "grub"
find_text "function"
find_code "dialogue"
```



**________________________________________________________________________________________________________________________________________________________________________________________________**

### **Enable Google-like search in terminal**

**Step 1: Install fzf**

*`sudo apt install fzf`*

**Step 2: Enable shell integration (THIS is the key)**

*`source /usr/share/doc/fzf/examples/key-bindings.bash`*

**Step 3: Make it permanent (important)**

*`nano ~/.bashrc`*

**Paste at the bottom**

*`[ -f /usr/share/doc/fzf/examples/key-bindings.bash ] && source /usr/share/doc/fzf/examples/key-bindings.bash`*

**Then reload:**

*`source ~/.bashrc`*

**________________________________________________________________________________________________________________________________________________________________________________________________**

### **Quick Shortcuts**

**xkill: Kill apps by dragging over them**

**1️⃣ Open:**

👉 `System Settings → Keyboard → Shortcuts`

**2️⃣ Go to:**

👉 `Custom Shortcuts`

**3️⃣ Click:**

👉 `“Add Custom Shortcut”`

Name: *`xkill`*

Command: *`xkill`*

**4️⃣ Bind the key**

Click the newly created shortcut.

Click `“Unassigned”`

Press: *Ctrl + K*

**________________________________________________________________________________________________________________________________________________________________________________________________**

### Taking Selective Screenshot

Name: *`screenshot`*

Command: *`gnome-screenshot -a`*

**________________________________________________________________________________________________________________________________________________________________________________________________**

### **ColorPicker - System-wide**

**1. Install `gpick`**

*`sudo apt install gpick`*

**2. Set Up the Shortcut (Direct to Crosshair)**

Go to `System Settings > Keyboard > Shortcuts > Custom Shortcuts`

Add New:

Name: `Color Picker`

Command: `gpick --pick`

Assign your key (e.g., `Ctrl+G`).

**________________________________________________________________________________________________________________________________________________________________________________________________**

### **Power Button**

Right-click on Desktop and choose `Create a new launcher here`

Name: `Power-Off`

Command: `systemctl poweroff`


**________________________________________________________________________________________________________________________________________________________________________________________________**

### **BIOS launcher**

Name: `BIOS`

Command: `pkexec systemctl reboot --firmware-setup`

**________________________________________________________________________________________________________________________________________________________________________________________________**

### **Copy File/Folder names**

`mkdir -p ~/.local/share/nemo/scripts`

```
cat > "$HOME/.local/share/nemo/scripts/Copy file names" <<'EOF'
#!/usr/bin/env bash
[ "$#" -eq 0 ] && exit 0
printf '%s\n' "${@##*/}" | xclip -selection clipboard
EOF
```
`chmod +x "$HOME/.local/share/nemo/scripts/Copy file names"`




**________________________________________________________________________________________________________________________________________________________________________________________________**

### **Genome System monitor**

`sudo apt install gnome-system-monitor`

`GTK_THEME=Adwaita:dark gnome-system-monitor`

`echo "alias mint='GTK_THEME=Adwaita:dark gnome-system-monitor'" >> ~/.bashrc`

**________________________________________________________________________________________________________________________________________________________________________________________________**

### **How to Kill an app**

From `soft` to `hard` kill 🔪

1. Use `btop` select the process app, press `k` to kill

2. Note the `PID` from `btop` and run `sudo kill -9 <PID>`

3. If it is managed by `systemd` and killing it returns `insufficient permissions`
    - `systemctl list-units --type=service | grep <app-name>`
    - see something like: `app-name.service`
    - `sudo systemctl stop app-name`
    - `sudo systemctl disable app-name`
4. Hard disable / mask it: `sudo systemctl mask app-name`

5. If you don’t need it at all: `sudo apt remove app-name*`

6. clean leftovers: `sudo apt autoremove`

7. Sometimes apps run as `user services`, not system ones.
    - `systemctl --user list-units | grep -i app-name`
    - if found `systemctl --user stop <service-name>`
    - `systemctl --user disable <service-name>`

**________________________________________________________________________________________________________________________________________________________________________________________________**